# Radiomics Feature Analysis

This notebook performs basic descriptive and statistical analysis for extracted radiomics features.

## What it does
- Loads a radiomics CSV.
- Infers radiomics feature columns.
- Produces descriptive summaries and missingness/variance views.
- Computes correlation diagnostics and highly correlated feature pairs.
- Runs two-group statistical testing (Welch t-test and Mann-Whitney U) with Benjamini-Hochberg FDR correction when a grouping column is configured.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

try:
    import seaborn as sns

    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

In [ ]:
# ---- User configuration ----
CSV_PATH = Path("../tests/data/nifti_index_radiomics.csv")
OUT_DIR = None  # Set to None to disable file outputs

# Optional group label for statistical testing (must have exactly 2 non-null groups)
# Example: GROUP_COLUMN = "accord_progression_6mois"
GROUP_COLUMN = None

ALPHA = 0.05
TOP_N_VARIANCE = 20
HIGH_CORR_THRESHOLD = 0.90

In [ ]:
def infer_radiomics_feature_columns(df: pd.DataFrame) -> list[str]:
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    feature_tokens = (
        "_original_",
        "_wavelet",
        "_log-sigma",
        "_square",
        "_squareroot",
        "_logarithm",
        "_exponential",
        "_gradient",
        "_lbp",
        "_firstorder_",
        "_shape_",
        "_glcm_",
        "_glrlm_",
        "_glszm_",
        "_gldm_",
        "_ngtdm_",
    )

    excluded_exact = {"_source_idx"}
    excluded_prefixes = ("mask_", "totalseg_")

    candidates = []
    for col in numeric_cols:
        c = str(col)
        low = c.lower()
        if c in excluded_exact:
            continue
        if any(low.startswith(prefix) for prefix in excluded_prefixes):
            continue
        if any(token in low for token in feature_tokens):
            candidates.append(c)

    return candidates


def benjamini_hochberg(pvalues: list[float]) -> np.ndarray:
    p = np.asarray(pvalues, dtype=float)
    n = p.size
    if n == 0:
        return np.array([], dtype=float)

    order = np.argsort(p)
    ranked = p[order]

    q_ranked = np.empty(n, dtype=float)
    running = 1.0
    for i in range(n - 1, -1, -1):
        rank = i + 1
        val = ranked[i] * n / rank
        running = min(running, val)
        q_ranked[i] = running

    q = np.empty(n, dtype=float)
    q[order] = np.clip(q_ranked, 0.0, 1.0)
    return q


def cohens_d(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return np.nan
    vx = x.var(ddof=1)
    vy = y.var(ddof=1)
    pooled = ((nx - 1) * vx + (ny - 1) * vy) / (nx + ny - 2)
    if pooled <= 0:
        return np.nan
    return (x.mean() - y.mean()) / np.sqrt(pooled)

In [ ]:
if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV not found: {CSV_PATH.resolve()}")

if OUT_DIR is not None:
    OUT_DIR = Path(OUT_DIR)
    OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV_PATH)
feature_cols = infer_radiomics_feature_columns(df)

if not feature_cols:
    raise ValueError(
        "No radiomics feature columns inferred. "
        "Set a valid CSV or adjust infer_radiomics_feature_columns()."
    )

features_df = df[feature_cols].replace([np.inf, -np.inf], np.nan)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Inferred radiomics features: {len(feature_cols):,}")
if OUT_DIR is None:
    print("Output saving: disabled (OUT_DIR=None)")
else:
    print(f"Output folder: {OUT_DIR.resolve()}")

In [ ]:
missing_pct = features_df.isna().mean() * 100

desc = features_df.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T

desc["missing_pct"] = missing_pct

desc["cv"] = np.where(
    desc["mean"].abs() > 1e-12, desc["std"] / desc["mean"].abs(), np.nan
)

desc = desc.sort_values("std", ascending=False)

print("Top features by standard deviation:")
display(desc.head(25))

if OUT_DIR is not None:
    desc.to_csv(OUT_DIR / "descriptive_summary.csv")
    print("Saved descriptive summary ->", OUT_DIR / "descriptive_summary.csv")

In [ ]:
missing_top = missing_pct.sort_values(ascending=False).head(40)

plt.figure(figsize=(12, 6))
missing_top.plot(kind="bar")
plt.ylabel("Missing (%)")
plt.title("Top 40 Features by Missingness")
plt.tight_layout()
plt.show()

In [ ]:
variance = features_df.var(skipna=True).sort_values(ascending=False)
top_var_features = variance.head(TOP_N_VARIANCE).index.tolist()

print("Top variance features:")
display(variance.head(TOP_N_VARIANCE).to_frame("variance"))

features_df[top_var_features].hist(figsize=(16, 12), bins=40, layout=(5, 4))
plt.suptitle("Distributions of Top Variance Features", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
corr_features = variance.head(min(30, len(variance))).index.tolist()
corr = features_df[corr_features].corr(method="spearman")

plt.figure(figsize=(14, 10))
if HAS_SEABORN:
    sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.2)
else:
    plt.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
    plt.colorbar(label="Spearman r")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90, fontsize=8)
    plt.yticks(range(len(corr.index)), corr.index, fontsize=8)
plt.title("Spearman Correlation (Top Variance Features)")
plt.tight_layout()
plt.show()

pairs = []
cols = corr.columns.tolist()
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        val = corr.iat[i, j]
        if np.isfinite(val) and abs(val) >= HIGH_CORR_THRESHOLD:
            pairs.append((cols[i], cols[j], float(val)))

corr_pairs = pd.DataFrame(
    pairs, columns=["feature_a", "feature_b", "spearman_r"]
).sort_values("spearman_r", key=lambda s: s.abs(), ascending=False)

display(corr_pairs.head(50))
if OUT_DIR is not None:
    corr_pairs.to_csv(OUT_DIR / "high_correlation_pairs.csv", index=False)
    print("Saved high-correlation pairs ->", OUT_DIR / "high_correlation_pairs.csv")

In [ ]:
if GROUP_COLUMN is None or GROUP_COLUMN not in df.columns:
    print(
        "Skipping inferential statistics: set GROUP_COLUMN to a valid binary label column."
    )
else:
    group_data = df[[GROUP_COLUMN] + feature_cols].copy()
    group_data = group_data.dropna(subset=[GROUP_COLUMN])
    groups = [g for g in group_data[GROUP_COLUMN].unique() if pd.notna(g)]

    if len(groups) != 2:
        print(
            f"Skipping inferential statistics: expected 2 groups, found {len(groups)} groups: {groups}"
        )
    else:
        g0, g1 = groups[0], groups[1]
        print(f"Running tests for groups: {g0!r} vs {g1!r}")

        records = []
        for col in feature_cols:
            x = (
                group_data.loc[group_data[GROUP_COLUMN] == g0, col]
                .dropna()
                .to_numpy(dtype=float)
            )
            y = (
                group_data.loc[group_data[GROUP_COLUMN] == g1, col]
                .dropna()
                .to_numpy(dtype=float)
            )

            if len(x) < 3 or len(y) < 3:
                continue

            ttest_p = stats.ttest_ind(x, y, equal_var=False, nan_policy="omit").pvalue
            try:
                mwu_p = stats.mannwhitneyu(x, y, alternative="two-sided").pvalue
            except ValueError:
                mwu_p = np.nan

            records.append(
                {
                    "feature": col,
                    "n_group_0": len(x),
                    "n_group_1": len(y),
                    "mean_group_0": float(np.mean(x)),
                    "mean_group_1": float(np.mean(y)),
                    "median_group_0": float(np.median(x)),
                    "median_group_1": float(np.median(y)),
                    "cohens_d": cohens_d(x, y),
                    "p_ttest_welch": float(ttest_p) if np.isfinite(ttest_p) else np.nan,
                    "p_mannwhitney": float(mwu_p) if np.isfinite(mwu_p) else np.nan,
                }
            )

        stats_df = pd.DataFrame(records)
        if stats_df.empty:
            print("No features had enough samples per group for testing.")
        else:
            stats_df["q_ttest_bh"] = benjamini_hochberg(
                stats_df["p_ttest_welch"].fillna(1.0).tolist()
            )
            stats_df["q_mannwhitney_bh"] = benjamini_hochberg(
                stats_df["p_mannwhitney"].fillna(1.0).tolist()
            )
            stats_df["sig_ttest"] = stats_df["q_ttest_bh"] < ALPHA
            stats_df["sig_mannwhitney"] = stats_df["q_mannwhitney_bh"] < ALPHA

            stats_df = stats_df.sort_values(
                ["q_mannwhitney_bh", "q_ttest_bh", "feature"]
            )
            display(stats_df.head(50))

            if OUT_DIR is not None:
                stats_df.to_csv(OUT_DIR / "group_stat_tests.csv", index=False)
                print("Saved inferential stats ->", OUT_DIR / "group_stat_tests.csv")

            top_sig = stats_df.loc[
                stats_df["sig_mannwhitney"] | stats_df["sig_ttest"]
            ].head(12)
            if not top_sig.empty:
                for f in top_sig["feature"].tolist():
                    plt.figure(figsize=(8, 4))
                    plot_df = group_data[[GROUP_COLUMN, f]].dropna()
                    if HAS_SEABORN:
                        sns.boxplot(data=plot_df, x=GROUP_COLUMN, y=f)
                    else:
                        vals0 = plot_df.loc[plot_df[GROUP_COLUMN] == g0, f].values
                        vals1 = plot_df.loc[plot_df[GROUP_COLUMN] == g1, f].values
                        plt.boxplot([vals0, vals1], labels=[str(g0), str(g1)])
                        plt.xlabel(GROUP_COLUMN)
                        plt.ylabel(f)
                    plt.title(f"{f} by {GROUP_COLUMN}")
                    plt.tight_layout()
                    plt.show()

## Notes

- This notebook infers feature columns heuristically; adjust `infer_radiomics_feature_columns()` if your naming differs.
- For large feature sets, focus on top-variance features or add feature filtering before inferential testing.
- Statistical tests here are univariate and exploratory.
